# DiD-BCF — B2_sweep (N=800, linearity_degree=1)  ·  part 4/5

**Workstream B2 · canonical DiD (selection on unobservables)

bias->0, var->0, sqrt(N) behaviour**

Fits DiD-BCF on this scenario at the **single sample size N above** and reports
metrics for both the plain DiD-BCF posterior and the proposed posterior
correction. The B2 sample-size sweep is split one-N-per-notebook to keep run
time manageable (combine the four N CSVs afterwards for the sqrt(N) analysis).

> **Colab:** upload just this notebook and *Run all*.


**This notebook runs PART 4 of 5** — replications 60–79 of 100 (seed = rep index). Run *all* 5 parts; each auto-downloads its own `summaries_B2_sweep_N800_lin_1_part4of5.csv`, and the parts concatenate back into the full 100-replication result (they match aggregate_metrics.py's `summaries_*.csv` glob).

In [ ]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

In [ ]:
REPS  = 100     # TOTAL replications for this (N, linearity_degree), summed across all parts
JOBS  = 1       # parallel reps (keep 1 on a single-core/GPU Colab)
N     = 800      # <-- this notebook's sample size (one of 200 / 400 / 800 / 1600)
LIN   = 1        # linearity degree for this notebook

# --- This notebook runs ONE PART of the 100 replications ----------------------
# The B2 sweep at large N is too slow to run all 100 reps in one Colab session,
# so each (N, LIN) is split into PARTS notebooks covering DISTINCT, non-
# overlapping reps (the seed IS the rep index). Run every part; each writes its
# own summaries_*_part{PART}of{PARTS}.csv. aggregate_metrics.py globs
# summaries_*.csv, so the parts concatenate back to the full 100-rep result.
PART  = 4        # <-- this part (1 .. PARTS)
PARTS = 5        # <-- number of parts this (N, LIN) is split into
assert REPS % PARTS == 0, "REPS must divide evenly into PARTS"
CHUNK     = REPS // PARTS
REP_START = (PART - 1) * CHUNK
REP_STOP  = PART * CHUNK            # this part runs reps [REP_START, REP_STOP)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

import pandas as pd
from did_bcf_revision import config as cfg
from did_bcf_revision.runner import process_rep

EXP_NAME = "B2_sweep"
exp = cfg.get_experiment(EXP_NAME)
params = dict(exp.dgp_params)
params["linearity_degree"] = LIN

rep_indices = range(REP_START, REP_STOP)
print(f"[{EXP_NAME} N={N} lin={LIN}] part {PART}/{PARTS}: reps "
      f"{REP_START}..{REP_STOP - 1}  ({len(rep_indices)} fits)")

def _one(rep):
    return process_rep(exp.dgp, params, N, rep, exp.name,
                       bcf_params=bcf_params, prop_method="logit", n_splits=2, spec="structured")

try:
    from tqdm import tqdm
    iterator = tqdm(rep_indices, desc=f"{EXP_NAME} N{N} p{PART}", unit="fit")
except Exception:
    iterator = rep_indices
rows = [_one(rep) for rep in iterator]

summaries = pd.concat(rows, ignore_index=True)
out_csv = f"summaries_{EXP_NAME}_N{N}_lin_{LIN}_part{PART}of{PARTS}.csv"
summaries.to_csv(out_csv, index=False)
print("wrote", out_csv, "| rows:", len(summaries), "| reps:", summaries["rep"].nunique())

# --- Auto-save & download so the run needs no supervision -------------------
# Download the CSV the moment the slow fits finish (before the preview-metric
# cells below), so an interrupted preview never loses the expensive results.
try:
    from google.colab import files
    files.download(out_csv)
    print("downloaded", out_csv)
except Exception as e:
    print("(not on Colab / download skipped):", e)

summaries.head()


In [ ]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [ ]:
surface_metrics(summaries)